# Spectral Microscope: Quickstart

This notebook demonstrates how to capture inline spectral telemetry out-of-the-box on a small model (`SmolLM2-1.7B-Instruct`).

> **Note:** We run this on CPU/Auto for demonstration purposes. If using Qwen models on CUDA, prefer `dtype=torch.bfloat16` in `from_pretrained`.

In [ ]:
!pip install -e .

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from prism import SpectralMicroscope, __version__

print(f"prism version: {__version__}")

# 1. Load Model (Eager Attention is mandatory!)
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    attn_implementation="eager"
)
print("Model loaded!")

### Run Spectral Microscope

We now attach the microscope and generate a short response. The tool will track spectral entropy and effective mathematical dimension at each autoregressive step.

In [ ]:
# 2. Initialize Telemetry
microscope = SpectralMicroscope(max_tokens=64, window_size=32, streaming_cov_alpha=0.95)

prompt = "Explain what a black hole is in three simple sentences."

print("Generating...")
result = microscope.generate_and_analyze(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=50,
    temperature=0.7
)

print("\n--- Output ---")
print(result["response"])
print("--------------\n")

### View Telemetry
Let's look at the captured trajectory. We'll load it into a pandas dataframe for easy viewing.

In [ ]:
import pandas as pd

df = pd.DataFrame(result["telemetry"])
df.head(15)

In [ ]:
# Quick visualization of the trajectory
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(df["step"], df["spectral_entropy"], marker='o', color='crimson')
plt.title("Inline Spectral Entropy Trajectory")
plt.xlabel("Generation Step")
plt.ylabel("Spectral Entropy (Bits)")
plt.grid(True, alpha=0.3)
plt.show()